In [2]:
# Cell 1: prepare environment and sys.path
import sys, os
repo_root = os.path.expanduser('~/Documents/Programming/SynRxNet/SynRxNet')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print('Added repo to sys.path:', repo_root)
import platform
print('Python:', platform.python_version())
print('Torch available:', end=' ')
try:
    import torch
    print(torch.__version__)
except Exception as e:
    print('not found -', e)

Added repo to sys.path: /Users/rishit/Documents/Programming/SynRxNet/SynRxNet
Python: 3.10.19
Torch available: 2.9.1


In [3]:
# Cell 2: imports and utilities
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pprint import pprint

# Local model imports (will raise if files not in place)
from src.models.baseline.mlp_baseline import MLPBaseline
from src.models.baseline.xgb_baseline import XGBBaseline

# XGBoost is optional in your environment; we'll try to import it and set a flag
try:
    import xgboost as xgb
    HAVE_XGB = True
    print('xgboost version:', xgb.__version__)
except Exception:
    HAVE_XGB = False
    print('xgboost not available; XGBBaseline test will be skipped or run conditionally')

xgboost version: 3.1.2


## Basic Testing

In [3]:
# --- synthetic dataset ---
torch.manual_seed(0)
N = 256
batch_size = 32
drug_dim = 128
cellline_dim = 64

xA = torch.randn(N, drug_dim)
xB = torch.randn(N, drug_dim)
cell = torch.randn(N, cellline_dim)
# simple synthetic regression target
y = (xA.mean(dim=1) + xB.mean(dim=1) + cell.mean(dim=1)) * 0.5

dataset = TensorDataset(xA, xB, cell, y)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print('Dataset shapes:', xA.shape, xB.shape, cell.shape, y.shape)
print('Batches per epoch:', len(loader))

Dataset shapes: torch.Size([256, 128]) torch.Size([256, 128]) torch.Size([256, 64]) torch.Size([256])
Batches per epoch: 8


In [4]:
# --- helper training loop ---
def train_model_torch(model, loader, epochs=3, lr=1e-3, device='cpu'):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for epoch in range(1, epochs+1):
        running = 0.0
        for step, batch in enumerate(loader):
            drugA_batch, drugB_batch, cell_batch, y_batch = [t.to(device) for t in batch]
            opt.zero_grad()
            preds = model(drugA_batch, drugB_batch, cell_batch)
            loss = loss_fn(preds, y_batch)
            loss.backward()
            opt.step()
            running += loss.item()
        print(f'Epoch {epoch} avg loss: {running / len(loader):.6f}')
    return model

In [5]:
# --- Train MLPBaseline ---
mlp = MLPBaseline(inpute_dims=(drug_dim, drug_dim, cellline_dim), hidden_dims=[64,32,16], dropout=0.1, lr=1e-3)
print('MLPBaseline instantiated with drug_dim=', mlp.drug_dim, 'cellline_dim=', mlp.cellline_dim)
mlp = train_model_torch(mlp, loader, epochs=3, lr=1e-3, device='cpu')

# Quick validation check
mlp.eval()
with torch.no_grad():
    sample = next(iter(loader))
    drugA_s, drugB_s, cell_s, y_s = [t for t in sample]
    preds = mlp(drugA_s, drugB_s, cell_s)
    pearson = mlp._pearson_correlation(preds, y_s)
    print('Validation sample pearson:', float(pearson))

MLPBaseline instantiated with drug_dim= 128 cellline_dim= 64
Epoch 1 avg loss: 0.039713
Epoch 2 avg loss: 0.027424
Epoch 3 avg loss: 0.019047
Validation sample pearson: 0.4491300582885742


In [6]:
# --- Optional: tiny XGBoost smoke test ---
if HAVE_XGB:
    print('Running lightweight XGBoost smoke test...')
    from sklearn.preprocessing import StandardScaler
    X = np.concatenate([xA.numpy(), xB.numpy(), cell.numpy()], axis=1)
    y_np = y.numpy()
    try:
        dtrain = xgb.DMatrix(X, label=y_np)
        xgb_params = {'objective': 'reg:squarederror', 'verbosity': 0, 'tree_method': 'hist'}
        xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=10)
        # instantiate wrapper and attach scaler + model
        xgb_wrapper = XGBBaseline(input_dims=(drug_dim, drug_dim, cellline_dim))
        scaler = StandardScaler().fit(X)
        xgb_wrapper.feature_scaler = scaler
        xgb_wrapper.xgb_model = xgb_model
        xgb_wrapper.is_fitted = True
        # Predict on a small sample:
        sample_X = X[:8]
        sample_drugA = torch.from_numpy(sample_X[:, :drug_dim]).float()
        sample_drugB = torch.from_numpy(sample_X[:, drug_dim:drug_dim*2]).float()
        sample_cell = torch.from_numpy(sample_X[:, drug_dim*2:]).float()
        preds = xgb_wrapper(sample_drugA, sample_drugB, sample_cell)
        print('XGB wrapper sample preds:', preds.cpu().numpy())
    except Exception as e:
        print('XGBoost smoke test failed:', e)
else:
    print('\\nSkipping XGBoost smoke test because xgboost is not installed.')

Running lightweight XGBoost smoke test...
XGB wrapper sample preds: [ 0.00505655 -0.08461982  0.06971977  0.06322438 -0.01047168 -0.01828492
 -0.052155    0.02250665]


## Final test before training

## Real-data: O'Neil (ZIP target) using ChemBERTa + CCLE PCA

In [7]:
# Cell: real-data setup
import os
import numpy as np
import pandas as pd
import torch
from src.datasets.feature_engineer import FeatureEngineer

# Paths (relative to repo root)
ONEIL_CSV = '../data/processed/cleaned_oneil.csv'
CCLE_PCA_CSV = '../data/processed/ccle_cell_line_pca_100.csv'
CHEMBERTA_CACHE = '../data/chemberta_cache'

print('Files exist?', os.path.exists(ONEIL_CSV), os.path.exists(CCLE_PCA_CSV))

# FeatureEngineer (will read cached embeddings from CHEMBERTA_CACHE)
fe = FeatureEngineer(cache_dir=CHEMBERTA_CACHE)

# Read datasets
df = pd.read_csv(ONEIL_CSV)
ccle_df = pd.read_csv(CCLE_PCA_CSV, index_col=0)
pc_cols = [c for c in ccle_df.columns if c.startswith('PC')]
ccle_dim = len(pc_cols)
print('ONeil rows:', len(df), 'unique SMILES cols:', ['smiles_drug1','smiles_drug2'])

Files exist? True True
ONeil rows: 22919 unique SMILES cols: ['smiles_drug1', 'smiles_drug2']


In [8]:
# Cell: load ChemBERTa embeddings for unique SMILES (uses cached .pt files)
sm_cols = ['smiles_drug1', 'smiles_drug2']
unique_smiles = pd.unique(df[sm_cols].values.ravel())
unique_smiles = [s for s in unique_smiles if isinstance(s, str) and s]
print('Unique SMILES found:', len(unique_smiles))

emb_map = {}
first_dim = None
for sm in unique_smiles:
    try:
        emb = fe.encode_smiles(sm)
        if isinstance(emb, torch.Tensor):
            arr = emb.cpu().numpy()
        else:
            arr = np.asarray(emb, dtype=np.float32)
        emb_map[sm] = arr
        if first_dim is None:
            first_dim = arr.shape[0]
    except Exception as e:
        print('Failed to encode', sm, e)

if first_dim is None:
    raise RuntimeError('No ChemBERTa embeddings could be loaded from cache')

print('ChemBERTa embedding dim:', first_dim)

Unique SMILES found: 38
ChemBERTa embedding dim: 768


In [9]:
# Helper: map cleaned_oneil cell-line name to CCLE PCA row (simple startswith match)
def map_cellline_to_pca(name):
    if not isinstance(name, str) or not name:
        return np.zeros(ccle_dim, dtype=np.float32)
    matches = [idx for idx in ccle_df.index if str(idx).startswith(str(name))]
    if matches:
        return ccle_df.loc[matches[0], pc_cols].values.astype(np.float32)
    # try case-insensitive contains as fallback
    matches = [idx for idx in ccle_df.index if str(name).lower() in str(idx).lower()]
    if matches:
        return ccle_df.loc[matches[0], pc_cols].values.astype(np.float32)
    return np.zeros(ccle_dim, dtype=np.float32)

# Quick check on mapping
sample_cells = df['Cell line'].unique()[:10] if 'Cell line' in df.columns else []
for c in sample_cells:
    print(c, '->', map_cellline_to_pca(c).shape)

A2058 -> (42,)
A2780 -> (42,)
A375 -> (42,)
A427 -> (42,)
CAOV3 -> (42,)
COLO320DM -> (42,)
DLD1 -> (42,)
EFM192B -> (42,)
ES2 -> (42,)
HCT116 -> (42,)


In [10]:
# Build feature matrix X and target y (drop rows with missing ZIP or SMILES)
rows = []
ys = []
missing = 0
for _, r in df.iterrows():
    s1 = r.get('smiles_drug1')
    s2 = r.get('smiles_drug2')
    if not (isinstance(s1, str) and isinstance(s2, str)):
        missing += 1
        continue
    if pd.isna(r.get('ZIP')):
        missing += 1
        continue
    e1 = emb_map.get(s1)
    e2 = emb_map.get(s2)
    if e1 is None or e2 is None:
        missing += 1
        continue
    cell_vec = map_cellline_to_pca(r.get('Cell line'))
    Xr = np.concatenate([e1, e2, cell_vec], axis=0).astype(np.float32)
    rows.append(Xr)
    ys.append(float(r.get('ZIP')))

print('Dropped rows (missing):', missing)
X = np.vstack(rows)
y = np.array(ys, dtype=np.float32)
print('Built X,y shapes:', X.shape, y.shape)

# Derive dims
emb_dim = first_dim
cell_dim = ccle_dim
print('Embedding dim (per-drug):', emb_dim, 'cell PC dim:', cell_dim)

Dropped rows (missing): 0
Built X,y shapes: (22919, 1578) (22919,)
Embedding dim (per-drug): 768 cell PC dim: 42


In [11]:
# Create splits (80/10/10)
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print('Train/Val/Test sizes:', len(X_train), len(X_val), len(X_test))

Train/Val/Test sizes: 18335 2292 2292


In [12]:
# Train MLPBaseline on the ChemBERTa+CCLE features (3 epochs smoke-run)
import torch
from torch.utils.data import TensorDataset, DataLoader
# assume MLPBaseline is already imported in Cell 2; if not, import here:
from src.models.baseline.mlp_baseline import MLPBaseline

# Prepare tensors: split X into drugA/drugB/cell parts
def split_features(Xarr, emb_dim, cell_dim):
    drugA = Xarr[:, :emb_dim]
    drugB = Xarr[:, emb_dim:emb_dim*2]
    cell = Xarr[:, emb_dim*2:emb_dim*2+cell_dim]
    return drugA, drugB, cell

dA_tr, dB_tr, c_tr = split_features(X_train, emb_dim, cell_dim)
dA_val, dB_val, c_val = split_features(X_val, emb_dim, cell_dim)
dA_test, dB_test, c_test = split_features(X_test, emb_dim, cell_dim)

train_ds = TensorDataset(torch.from_numpy(dA_tr).float(), torch.from_numpy(dB_tr).float(), torch.from_numpy(c_tr).float(), torch.from_numpy(y_train).float())
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

# instantiate model with inferred dims
mlp = MLPBaseline(inpute_dims=(emb_dim, emb_dim, cell_dim), hidden_dims=[512,256,128], dropout=0.1, lr=1e-3)
print('MLP instantiated with input dims:', emb_dim, emb_dim, cell_dim)

# Reuse train_model_torch from earlier cells (if defined). If not, define a minimal trainer:
try:
    train_model_torch
except NameError:
    def train_model_torch(model, loader, epochs=3, lr=1e-3, device='cpu'):
        model.to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = torch.nn.MSELoss()
        for epoch in range(1, epochs+1):
            running = 0.0
            for step, batch in enumerate(loader):
                drugA_batch, drugB_batch, cell_batch, y_batch = [t.to(device) for t in batch]
                opt.zero_grad()
                preds = model(drugA_batch, drugB_batch, cell_batch)
                loss = loss_fn(preds, y_batch)
                loss.backward()
                opt.step()
                running += loss.item()
            print(f'Epoch {epoch} avg loss: {running / len(loader):.6f}')
        return model

# Train (3 epoch smoke-run)
mlp = train_model_torch(mlp, train_loader, epochs=3, lr=1e-3, device='cpu')

# Quick eval on test sample
mlp.eval()
with torch.no_grad():
    ta = torch.from_numpy(dA_test).float()[:256]
    tb = torch.from_numpy(dB_test).float()[:256]
    tc = torch.from_numpy(c_test).float()[:256]
    ty = torch.from_numpy(y_test).float()[:256]
    preds = mlp(ta, tb, tc)
    try:
        pear = mlp._pearson_correlation(preds, ty)
        print('Test sample Pearson:', float(pear))
    except Exception:
        print('Pearson calc unavailable')

MLP instantiated with input dims: 768 768 42
Epoch 1 avg loss: 338.672026
Epoch 2 avg loss: 310.117889
Epoch 3 avg loss: 308.072383
Test sample Pearson: 0.5077220797538757


In [13]:
# Optional: XGBoost baseline (if available)
try:
    import xgboost as xgb
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import mean_squared_error
    HAVE_XGB = True
except Exception:
    HAVE_XGB = False

if HAVE_XGB:
    print('Training XGBoost baseline...')
    scaler = StandardScaler().fit(X_train)
    dtrain = xgb.DMatrix(scaler.transform(X_train), label=y_train)
    dval = xgb.DMatrix(scaler.transform(X_val), label=y_val)
    watch = [(dtrain, 'train'), (dval, 'val')]
    params = {'objective': 'reg:squarederror', 'verbosity': 0, 'tree_method': 'hist'}
    xgb_model = xgb.train(params, dtrain, num_boost_round=100, evals=watch, early_stopping_rounds=10)
    preds_test = xgb_model.predict(xgb.DMatrix(scaler.transform(X_test)))
    rmse = np.sqrt(mean_squared_error(y_test, preds_test))
    print('XGBoost Test RMSE:', rmse)
else:
    print('Skipping XGBoost: xgboost not available in this environment')

Training XGBoost baseline...
[0]	train-rmse:18.42609	val-rmse:17.88472
[1]	train-rmse:17.33545	val-rmse:16.87840
[2]	train-rmse:16.60964	val-rmse:16.25473
[3]	train-rmse:16.10440	val-rmse:15.85663
[4]	train-rmse:15.68215	val-rmse:15.56253
[5]	train-rmse:15.38864	val-rmse:15.35716
[6]	train-rmse:15.13891	val-rmse:15.22857
[7]	train-rmse:14.96590	val-rmse:15.15305
[8]	train-rmse:14.73766	val-rmse:14.99000
[9]	train-rmse:14.49108	val-rmse:14.84679
[10]	train-rmse:14.30913	val-rmse:14.79689
[11]	train-rmse:14.16536	val-rmse:14.76027
[12]	train-rmse:14.03436	val-rmse:14.67215
[13]	train-rmse:13.89500	val-rmse:14.61031
[14]	train-rmse:13.75551	val-rmse:14.51024
[15]	train-rmse:13.65085	val-rmse:14.49717
[16]	train-rmse:13.46470	val-rmse:14.37953
[17]	train-rmse:13.37617	val-rmse:14.32067
[18]	train-rmse:13.22613	val-rmse:14.23411
[19]	train-rmse:13.07718	val-rmse:14.13312
[20]	train-rmse:12.92740	val-rmse:14.03006
[21]	train-rmse:12.81795	val-rmse:13.94064
[22]	train-rmse:12.66071	val-rmse:1

### Notes
- This notebook uses cached ChemBERTa embeddings from `data/chemberta_cache`. If you prefer Morgan fingerprints instead, compute them with RDKit and replace the drug feature portion.
- Mapping from `Cell line` to CCLE PCA uses a `startswith` heuristic; adjust if you need exact mapping.
- The MLP training here is a short smoke-run (3 epochs). Increase epochs and add proper validation logging for experiments.

In [17]:
# Use repo dataset classes to get ready-to-train loaders (ChemBERTa + CCLE PCA, ZIP target)
import torch
from torch.utils.data import DataLoader
import numpy as np

from src.datasets.feature_engineer import FeatureEngineer
from src.datasets.splitter import Splitter
from src.datasets.multimodal_dataset import MultimodalSynergyDataset

# Paths
CSV = "../data/processed/cleaned_oneil.csv"
CCLE_PCA = "../data/processed/ccle_cell_line_pca_100.csv"
CHEMBERTA_CACHE = "../data/chemberta_cache"

# Init helpers
fe = FeatureEngineer(cache_dir=CHEMBERTA_CACHE)
splitter = Splitter(strategy="random", seed=42)

# Create dataset objects (they will validate SMILES, attach cell-line PCA, and apply split)
train_ds = MultimodalSynergyDataset(
    csv_path=CSV,
    splitter=splitter,
    feature_engineer=fe,
    subset="train",
    cell_line_features_path=CCLE_PCA,
    n_cell_line_components=100,
    missing_cell_policy="drop",
)

val_ds = MultimodalSynergyDataset(
    csv_path=CSV,
    splitter=splitter,
    feature_engineer=fe,
    subset="val",
    cell_line_features_path=CCLE_PCA,
    n_cell_line_components=100,
    missing_cell_policy="drop",
)

test_ds = MultimodalSynergyDataset(
    csv_path=CSV,
    splitter=splitter,
    feature_engineer=fe,
    subset="test",
    cell_line_features_path=CCLE_PCA,
    n_cell_line_components=100,
    missing_cell_policy="drop",
)

print("sizes (train/val/test):", len(train_ds), len(val_ds), len(test_ds))

# Collate: pull chemberta embeddings and cell PCA, and ZIP target
def collate_chemberta_batch(batch):
    # batch is a list of dicts returned by MultimodalSynergyDataset.__getitem__
    drug1 = torch.stack([b["drug1_chemberta"] for b in batch])
    drug2 = torch.stack([b["drug2_chemberta"] for b in batch])
    cell  = torch.stack([b["cell_line"] for b in batch])
    # targets is a tensor [ZIP, Bliss, Loewe, HSA]; we take ZIP (index 0)
    targets = torch.stack([b["targets"] for b in batch])[:, 0]
    return drug1, drug2, cell, targets

# DataLoaders ready to feed into MLPBaseline or XGB wrapper
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_chemberta_batch)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, collate_fn=collate_chemberta_batch)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, collate_fn=collate_chemberta_batch)

# Example: inspect a single batch
drugA, drugB, cell, y = next(iter(train_loader))
print("batch shapes:", drugA.shape, drugB.shape, cell.shape, y.shape)
# ready for MLPBaseline(drug_dim, drug_dim, cell_dim)

7 cell lines in synergy data could not be matched to PCA features: ['COLO320DM', 'DLD1', 'EFM192B', 'OCUBM', 'PA1', 'UWB1289', 'UWB1289BRCA1']
Dropping 4141 rows due to missing cell-line features (after mapping).
7 cell lines in synergy data could not be matched to PCA features: ['COLO320DM', 'DLD1', 'EFM192B', 'OCUBM', 'PA1', 'UWB1289', 'UWB1289BRCA1']
Dropping 4141 rows due to missing cell-line features (after mapping).
7 cell lines in synergy data could not be matched to PCA features: ['COLO320DM', 'DLD1', 'EFM192B', 'OCUBM', 'PA1', 'UWB1289', 'UWB1289BRCA1']
Dropping 4141 rows due to missing cell-line features (after mapping).
7 cell lines in synergy data could not be matched to PCA features: ['COLO320DM', 'DLD1', 'EFM192B', 'OCUBM', 'PA1', 'UWB1289', 'UWB1289BRCA1']
Dropping 4141 rows due to missing cell-line features (after mapping).
7 cell lines in synergy data could not be matched to PCA features: ['COLO320DM', 'DLD1', 'EFM192B', 'OCUBM', 'PA1', 'UWB1289', 'UWB1289BRCA1']
Dropp

sizes (train/val/test): 15022 1878 1878
batch shapes: torch.Size([64, 768]) torch.Size([64, 768]) torch.Size([64, 42]) torch.Size([64])
